# Testing `EntityFusionPolars` on the PPP dataset

Exercises `Entity_Fusion_Polars.py` against `test_data/100k.csv` (a PPP loan extract — despite the filename it's actually ~968k rows, so this doubles as a scale check against the ~1M-row target).

We treat `BorrowerName` + `BorrowerAddress` + `BorrowerZip` as exact-match identifiers with `k_required=2` (a pair must agree on at least 2 of the 3 columns to link) and walk through: the pre-clustering block-size check, timing the full run, and inspecting `cluster_stats` for merges worth reviewing.

In [1]:
import time
import polars as pl

from Entity_Fusion_Polars import EntityFusionPolars

pl.Config.set_tbl_rows(20)

polars.config.Config

## Load & normalize

Exact matching has no tolerance for casing/whitespace drift the way cosine matching does, so the identifier columns need normalizing before they're usable as exact-match keys.

In [2]:
df = pl.read_csv("test_data/100k.csv", infer_schema_length=5000)
df = df.with_row_index("record_id").with_columns(pl.col("record_id").cast(pl.Utf8))
df = df.with_columns(
    [
        pl.col("BorrowerName").str.strip_chars().str.to_uppercase(),
        pl.col("BorrowerAddress").str.strip_chars().str.to_uppercase(),
        pl.col("BorrowerZip").cast(pl.Utf8).str.strip_chars(),
    ]
)

print(f"{df.height:,} rows")
df.select("record_id", "BorrowerName", "BorrowerAddress", "BorrowerZip", "BorrowerCity").head()

968,525 rows


record_id,BorrowerName,BorrowerAddress,BorrowerZip,BorrowerCity
str,str,str,str,str
"""0""","""SUMTER COATINGS, INC.""","""2410 HIGHWAY 15 SOUTH""","""29150-9662""","""Sumter"""
"""1""","""PLEASANT PLACES, INC.""","""7684 SOUTHRAIL ROAD""","""29420-9000""","""North Charleston"""
"""2""","""BOYER CHILDREN'S CLINIC""","""1850 BOYER AVE E""","""98112-2922""","""SEATTLE"""
"""3""","""KIRTLEY CONSTRUCTION INC""","""1661 MARTIN RANCH RD""","""92407-1740""","""SAN BERNARDINO"""
"""4""","""AERO BOX LLC""",null,null,null


## Pre-clustering block-size check

Run this before the expensive part: for each match column, how many distinct rows share the same value? A runaway value (a placeholder address, a common name) would blow up the pairwise self-join downstream — better to see it here first.

In [21]:
linker = EntityFusionPolars(
    record_id_col="record_id",
    match_columns=[
        "BorrowerAddress",
        # "BorrowerZip",
        {"column": "BorrowerName", "block_on": ["BorrowerState"]},
    ],
    k_required=2,
    block_cap=2000,
)

t0 = time.time()
blocks = linker.preview_blocks(df)
print(f"preview_blocks: {time.time() - t0:.2f}s")
print(f"largest block: {blocks['n_hash_ids'].max()} (block_cap={linker.block_cap})")
blocks.head(10)

preview_blocks: 0.71s
largest block: 92 (block_cap=2000)


match_column,block,value,n_hash_ids
str,str,str,u32
"""BorrowerAddress""","""""","""PO BOX""",92
"""BorrowerAddress""","""""","""1140 RESERVOIR AVE""",48
"""BorrowerAddress""","""""","""2390 TOWER DR""",48
"""BorrowerAddress""","""""","""11410 COMMON OAKS DR""",43
"""BorrowerAddress""","""""","""TBD""",42
"""BorrowerAddress""","""""","""830 W TRAILCREEK DR""",41
"""BorrowerAddress""","""""","""7 PEARL CT""",41
"""BorrowerAddress""","""""","""14504 HERTZ QUAIL SPRINGS PKWY""",37
"""BorrowerAddress""","""""","""807 DORSEY STREET""",36


The largest block is well under `block_cap`, so nothing gets excluded here — safe to run the full pipeline.

## Run clustering

In [4]:
manual_address_clusters = pl.DataFrame({
    "manual_cluster_id": ["ADDR_ALIAS_001", "ADDR_ALIAS_001"],
    "field": ["BorrowerAddress", "BorrowerAddress"],
    "value": ["7 PEARL CT", "11410 COMMON OAKS DR"],
    "active": [True, True],
})

out = linker.cluster(
    df,
    manual_identifier_clusters=manual_address_clusters,
)



In [5]:
t0 = time.time()
out = linker.cluster(df)
elapsed = time.time() - t0
print(f"cluster(): {elapsed:.2f}s for {df.height:,} rows")

records, stats = out["records"], out["cluster_stats"]
print(f"{stats.height:,} clusters total")

cluster(): 9.06s for 968,525 rows
819,527 clusters total


## Manual decision smoke test

Create a tiny persistent review table and force records `396923` and `538365` into the same component with a `must_link` decision. The baseline `records`/`stats` outputs above stay unchanged; `manual_out` shows the override behavior separately.


In [6]:
manual_decision_records = ["396923", "538365"]

manual_decisions = pl.DataFrame({
    "record_id_a": [manual_decision_records[0]],
    "record_id_b": [manual_decision_records[1]],
    "decision": ["must_link"],
    "reviewer": ["notebook_test"],
    "reason": ["manual source-of-truth smoke test"],
    "active": [True],
})

baseline_pair = (
    records
    .filter(pl.col("record_id").is_in(manual_decision_records))
    .select("record_id", "BorrowerName", "BorrowerAddress", "BorrowerZip", "hash_id", "cluster_id", "stable_cluster_id")
    .sort("record_id")
)

t0 = time.time()
manual_out = linker.cluster(df, manual_decisions=manual_decisions)
manual_elapsed = time.time() - t0

manual_records = manual_out["records"]
manual_pair = (
    manual_records
    .filter(pl.col("record_id").is_in(manual_decision_records))
    .select("record_id", "BorrowerName", "BorrowerAddress", "BorrowerZip", "hash_id", "cluster_id", "stable_cluster_id")
    .sort("record_id")
)

print(f"manual cluster(): {manual_elapsed:.2f}s")
print("baseline cluster ids:")
print(baseline_pair)
print("manual-decision cluster ids:")
print(manual_pair)
print("manual_edges:")
print(manual_out["manual_edges"])
print("manual_conflicts:")
print(manual_out["manual_conflicts"])

assert manual_pair["cluster_id"].n_unique() == 1


manual cluster(): 9.23s
baseline cluster ids:
shape: (2, 7)
┌───────────┬──────────────┬──────────────┬─────────────┬──────────────┬────────────┬──────────────┐
│ record_id ┆ BorrowerName ┆ BorrowerAddr ┆ BorrowerZip ┆ hash_id      ┆ cluster_id ┆ stable_clust │
│ ---       ┆ ---          ┆ ess          ┆ ---         ┆ ---          ┆ ---        ┆ er_id        │
│ str       ┆ str          ┆ ---          ┆ str         ┆ u64          ┆ str        ┆ ---          │
│           ┆              ┆ str          ┆             ┆              ┆            ┆ str          │
╞═══════════╪══════════════╪══════════════╪═════════════╪══════════════╪════════════╪══════════════╡
│ 396923    ┆ ALBUQUERQUE  ┆ 2390 TOWER   ┆ 71201-5760  ┆ 168105983596 ┆ 1167       ┆ AUTO_1046458 │
│           ┆ HOSPITALITY  ┆ DR           ┆             ┆ 10691656     ┆            ┆ 05202085965  │
│           ┆ LLC          ┆              ┆             ┆              ┆            ┆              │
│ 538365    ┆ APPLE HOTEL  ┆ 11

## Manual identifier-cluster smoke test

This uses `BorrowerName` as the identifier field because this PPP notebook does not include EIN. The pattern is the same for EIN: put the identifier values in the same `manual_cluster_id`, and every current row carrying either value joins the same component.


### EIN-shaped toy example

This is the intended source-of-truth shape for identifier aliases: two EIN values share one `manual_cluster_id`, so all records carrying either EIN join the same component.


In [7]:
ein_demo = pl.DataFrame([
    {"record_id": "E1A", "ein": "12-3456789", "phone": "P1"},
    {"record_id": "E1B", "ein": "12-3456789", "phone": "P2"},
    {"record_id": "E2A", "ein": "98-7654321", "phone": "P3"},
    {"record_id": "E2B", "ein": "98-7654321", "phone": "P4"},
    {"record_id": "E3A", "ein": "11-1111111", "phone": "P5"},
])

ein_linker = EntityFusionPolars(
    record_id_col="record_id",
    match_columns=["ein", "phone"],
    k_required=1,
)

ein_baseline = ein_linker.cluster(ein_demo)

manual_ein_identifier_clusters = pl.DataFrame({
    "manual_cluster_id": ["EIN_ALIAS_001", "EIN_ALIAS_001"],
    "field": ["ein", "ein"],
    "value": ["12-3456789", "98-7654321"],
    "active": [True, True],
})

ein_manual = ein_linker.cluster(
    ein_demo,
    manual_identifier_clusters=manual_ein_identifier_clusters,
)

print("manual_ein_identifier_clusters:")
print(manual_ein_identifier_clusters)
print("baseline EIN clusters:")
print(ein_baseline["records"].select("record_id", "ein", "cluster_id", "stable_cluster_id").sort("record_id"))
print("manual EIN alias clusters:")
print(ein_manual["records"].select("record_id", "ein", "cluster_id", "stable_cluster_id").sort("record_id"))
print("manual_identifier_edges:")
print(ein_manual["manual_identifier_edges"])

assert (
    ein_manual["records"]
    .filter(pl.col("ein").is_in(["12-3456789", "98-7654321"]))["cluster_id"]
    .n_unique()
    == 1
)


manual_ein_identifier_clusters:
shape: (2, 4)
┌───────────────────┬───────┬────────────┬────────┐
│ manual_cluster_id ┆ field ┆ value      ┆ active │
│ ---               ┆ ---   ┆ ---        ┆ ---    │
│ str               ┆ str   ┆ str        ┆ bool   │
╞═══════════════════╪═══════╪════════════╪════════╡
│ EIN_ALIAS_001     ┆ ein   ┆ 12-3456789 ┆ true   │
│ EIN_ALIAS_001     ┆ ein   ┆ 98-7654321 ┆ true   │
└───────────────────┴───────┴────────────┴────────┘
baseline EIN clusters:
shape: (5, 4)
┌───────────┬────────────┬────────────────────────┬───────────────────────────┐
│ record_id ┆ ein        ┆ cluster_id             ┆ stable_cluster_id         │
│ ---       ┆ ---        ┆ ---                    ┆ ---                       │
│ str       ┆ str        ┆ str                    ┆ str                       │
╞═══════════╪════════════╪════════════════════════╪═══════════════════════════╡
│ E1A       ┆ 12-3456789 ┆ 1                      ┆ AUTO_10409277717357437705 │
│ E1B       ┆ 12-3456

In [8]:
manual_address_clusters = pl.DataFrame({
    "manual_cluster_id": ["ADDR_ALIAS_001", "ADDR_ALIAS_001"],
    "field": ["BorrowerAddress", "BorrowerAddress"],
    "value": ["7 PEARL CT", "11410 COMMON OAKS DR"],
    "active": [True, True],
})

address_out = linker.cluster(
    df,
    manual_identifier_clusters=manual_address_clusters,
)

In [9]:
address_out['records'].head()

record_id,LoanNumber,DateApproved,SBAOfficeCode,ProcessingMethod,BorrowerName,BorrowerAddress,BorrowerCity,BorrowerState,BorrowerZip,LoanStatusDate,LoanStatus,Term,SBAGuarantyPercentage,InitialApprovalAmount,CurrentApprovalAmount,UndisbursedAmount,FranchiseName,ServicingLenderLocationID,ServicingLenderName,ServicingLenderAddress,ServicingLenderCity,ServicingLenderState,ServicingLenderZip,RuralUrbanIndicator,HubzoneIndicator,LMIIndicator,BusinessAgeDescription,ProjectCity,ProjectCountyName,ProjectState,ProjectZip,CD,JobsReported,NAICSCode,Race,Ethnicity,UTILITIES_PROCEED,PAYROLL_PROCEED,MORTGAGE_INTEREST_PROCEED,RENT_PROCEED,REFINANCE_EIDL_PROCEED,HEALTH_CARE_PROCEED,DEBT_INTEREST_PROCEED,BusinessType,OriginatingLenderLocationID,OriginatingLender,OriginatingLenderCity,OriginatingLenderState,Gender,Veteran,NonProfit,ForgivenessAmount,ForgivenessDate,hash_id,cluster_id,stable_cluster_id
str,i64,str,i64,str,str,str,str,str,str,str,str,i64,i64,f64,f64,f64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,str,i64,str,str,str,str,str,str,f64,str,u64,str,str
"""0""",9547507704,"""05/01/2020""",464,"""PPP""","""SUMTER COATINGS, INC.""","""2410 HIGHWAY 15 SOUTH""","""Sumter""",null,"""29150-9662""","""12/18/2020""","""Paid in Full""",24,100,769358.78,769358.78,0.0,null,19248,"""Synovus Bank""","""1148 Broadway""","""COLUMBUS""","""GA""","""31901-2429""","""U""","""N""","""N""","""Existing or more than 2 years …","""Sumter""","""SUMTER""","""SC""","""29150-9662""","""SC-05""",62.0,325510.0,"""Unanswered""","""Unknown/NotStated""",null,769358.78,null,null,null,null,null,"""Corporation""",19248,"""Synovus Bank""","""COLUMBUS""","""GA""","""Unanswered""","""Unanswered""",null,773553.37,"""11/20/2020""",451537429749584847,"""UN_451537429749584847""","""UN_451537429749584847"""
"""1""",9777677704,"""05/01/2020""",464,"""PPP""","""PLEASANT PLACES, INC.""","""7684 SOUTHRAIL ROAD""","""North Charleston""",null,"""29420-9000""","""09/28/2021""","""Paid in Full""",24,100,736927.79,736927.79,0.0,null,19248,"""Synovus Bank""","""1148 Broadway""","""COLUMBUS""","""GA""","""31901-2429""","""U""","""Y""","""Y""","""Existing or more than 2 years …","""North Charleston""","""CHARLESTON""","""SC""","""29420-9000""","""SC-06""",73.0,561730.0,"""White""","""Unknown/NotStated""",null,736927.79,null,null,null,null,null,"""Sole Proprietorship""",19248,"""Synovus Bank""","""COLUMBUS""","""GA""","""Male Owned""","""Non-Veteran""",null,746336.24,"""08/12/2021""",7860692010340941238,"""UN_7860692010340941238""","""UN_7860692010340941238"""
"""2""",5791407702,"""05/01/2020""",1013,"""PPP""","""BOYER CHILDREN'S CLINIC""","""1850 BOYER AVE E""","""SEATTLE""",null,"""98112-2922""","""03/17/2021""","""Paid in Full""",24,100,691355.0,691355.0,0.0,null,9551,"""Bank of America, National Asso…","""100 N Tryon St, Ste 170""","""CHARLOTTE""","""NC""","""28202-4024""","""U""","""N""","""N""","""New Business or 2 years or les…","""SEATTLE""","""KING""","""WA""","""98112-2922""","""WA-07""",75.0,null,"""Unanswered""","""Unknown/NotStated""",null,691355.0,null,null,null,null,null,"""Non-Profit Organization""",9551,"""Bank of America, National Asso…","""CHARLOTTE""","""NC""","""Unanswered""","""Unanswered""","""Y""",696677.49,"""02/10/2021""",7112994111882087739,"""UN_7112994111882087739""","""UN_7112994111882087739"""
"""3""",6223567700,"""05/01/2020""",920,"""PPP""","""KIRTLEY CONSTRUCTION INC""","""1661 MARTIN RANCH RD""","""SAN BERNARDINO""",null,"""92407-1740""","""10/16/2021""","""Paid in Full""",24,100,499871.0,499871.0,0.0,null,9551,"""Bank of America, National Asso…","""100 N Tryon St, Ste 170""","""CHARLOTTE""","""NC""","""28202-4024""","""U""","""N""","""N""","""New Business or 2 years or les…","""SAN BERNARDINO""","""SAN BERNARDINO""","""CA""","""92407-1740""","""CA-23""",21.0,236115.0,"""American Indian or Alaska Nati…","""Not Hispanic or Latino""",null,499871.0,null,null,null,null,null,"""Corporation""",9551,

In [10]:
# identifier_alias_records = ["396923", "538365"]

# identifier_alias_values = (
#     records
#     .filter(pl.col("record_id").is_in(identifier_alias_records))
#     .select("record_id", "BorrowerName")
#     .sort("record_id")
# )

# manual_identifier_clusters = identifier_alias_values.select(
#     manual_cluster_id=pl.lit("BORROWER_NAME_ALIAS_001"),
#     field=pl.lit("BorrowerName"),
#     value=pl.col("BorrowerName"),
#     reviewer=pl.lit("notebook_test"),
#     reason=pl.lit("manual identifier-alias smoke test"),
#     active=pl.lit(True),
# )

# alias_values = manual_identifier_clusters["value"].to_list()
# baseline_identifier_alias_records = (
#     records
#     .filter(pl.col("BorrowerName").is_in(alias_values))
#     .select("record_id", "BorrowerName", "BorrowerAddress", "BorrowerZip", "hash_id", "cluster_id", "stable_cluster_id")
#     .sort(["BorrowerName", "record_id"])
# )

# t0 = time.time()
# identifier_alias_out = linker.cluster(
#     df,
#     manual_identifier_clusters=manual_identifier_clusters,
# )
# identifier_alias_elapsed = time.time() - t0

# identifier_alias_records_out = (
#     identifier_alias_out["records"]
#     .filter(pl.col("BorrowerName").is_in(alias_values))
#     .select("record_id", "BorrowerName", "BorrowerAddress", "BorrowerZip", "hash_id", "cluster_id", "stable_cluster_id")
#     .sort(["BorrowerName", "record_id"])
# )

# print(f"manual identifier cluster(): {identifier_alias_elapsed:.2f}s")
# print("manual_identifier_clusters source table:")
# print(manual_identifier_clusters)
# print("baseline affected records:")
# print(baseline_identifier_alias_records)
# print("manual-identifier affected records:")
# print(identifier_alias_records_out)
# print("manual_identifier_edges:")
# print(identifier_alias_out["manual_identifier_edges"])

# assert identifier_alias_records_out["cluster_id"].n_unique() == 1
# assert identifier_alias_out["manual_identifier_edges"].height >= 1


## Overall summary

Split clusters into real matches (`cluster_id` is a plain integer) vs untouched singletons (`UN_` prefix).

In [11]:
matched = stats.filter(~pl.col("cluster_id").str.starts_with("UN_"))
singletons = stats.filter(pl.col("cluster_id").str.starts_with("UN_"))

print(f"matched clusters (size >= 2): {matched.height:,}")
print(f"untouched singletons:         {singletons.height:,}")
print(f"rows in matched clusters:     {matched['n_primary_ids'].sum():,}")
print(f"rows untouched:               {singletons['n_primary_ids'].sum():,}")
print(f"overall dedup_ratio (matched only), mean: {matched['dedup_ratio'].mean():.2f}")

matched clusters (size >= 2): 97,513
untouched singletons:         722,014
rows in matched clusters:     220,471
rows untouched:               748,054
overall dedup_ratio (matched only), mean: 1.01


## Largest clusters

Sorted by row count. Worth eyeballing the top of this list before trusting it — a big cluster isn't automatically a good one.

In [12]:
matched.sort("n_primary_ids", descending=True).head(10)

cluster_id,stable_cluster_id,n_primary_ids,n_hash_ids,n_edges,thin_edges,n_manual_edges,dedup_ratio,edge_density,single_edge_cluster,n_distinct_BorrowerAddress,n_distinct_BorrowerZip,n_distinct_BorrowerName
str,str,u32,u32,u32,u32,u32,f64,f64,bool,u32,u32,u32
"""2519""","""AUTO_221585595102524726""",56,48,910,910,0,1.166667,0.052747,false,3,1,45
"""1167""","""AUTO_104645805202085965""",50,49,997,997,0,1.020408,0.049147,false,3,1,45
"""1690""","""AUTO_149907957114869028""",49,49,602,602,0,1.0,0.081395,false,1,2,35
"""3464""","""AUTO_305120516945188409""",48,48,559,559,0,1.0,0.085868,false,1,2,41
"""4397""","""AUTO_388282515727979673""",45,43,903,903,0,1.046512,0.047619,false,1,1,43
"""6872""","""AUTO_597471602931765632""",42,41,820,820,0,1.02439,0.05,false,1,1,41
"""8972""","""AUTO_792011735008747999""",42,42,397,397,0,1.0,0.105793,false,3,2,27
"""16851""","""AUTO_1522889830373392479""",41,40,670,670,0,1.025,0.059701,false,2,1,39
"""867""","""AUTO_78811878241398370""",37,37,266,266,0,1.0,0.139098,false,5,1,23


## Inspecting the top cluster

Look at `n_distinct_BorrowerName` vs `n_distinct_BorrowerAddress`/`n_distinct_BorrowerZip` for the largest cluster: if the name count is high while address/zip count is low, this cluster is being held together by a shared address/zip across many *differently named* borrowers — plausible for a shared mailbox address, a franchise HQ, or an office building, but not necessarily the same real-world entity. That's exactly the kind of merge this review table exists to surface.

In [13]:
top_cluster_id = matched.sort("n_primary_ids", descending=True)[0, "cluster_id"]
top_row = matched.filter(pl.col("cluster_id") == top_cluster_id)
print(top_row)

sample = (
    records.filter(pl.col("cluster_id") == top_cluster_id)
    .select("record_id", "BorrowerName", "BorrowerAddress", "BorrowerZip")
    .unique(subset=["BorrowerName", "BorrowerAddress", "BorrowerZip"])
    .sort("BorrowerName")
)
print(f"\n{sample.height} distinct name/address/zip combos in cluster {top_cluster_id}:")
sample.head(15)

shape: (1, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ cluster_i ┆ stable_cl ┆ n_primary ┆ n_hash_id ┆ … ┆ single_ed ┆ n_distinc ┆ n_distinc ┆ n_distin │
│ d         ┆ uster_id  ┆ _ids      ┆ s         ┆   ┆ ge_cluste ┆ t_Borrowe ┆ t_Borrowe ┆ ct_Borro │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ r         ┆ rAddress  ┆ rZip      ┆ werName  │
│ str       ┆ str       ┆ u32       ┆ u32       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ bool      ┆ u32       ┆ u32       ┆ u32      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2519      ┆ AUTO_2215 ┆ 56        ┆ 48        ┆ … ┆ false     ┆ 3         ┆ 1         ┆ 45       │
│           ┆ 855951025 ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│           ┆ 24726     ┆           ┆           ┆   ┆           ┆           

record_id,BorrowerName,BorrowerAddress,BorrowerZip
str,str,str,str
"""538365""","""APPLE HOTEL HOLDINGS LLC""","""11410 COMMON OAKS DR""","""27614-7002"""
"""545074""","""APPLE HOTEL HOLDINGS, LLC""","""11410 COMMON OAKS DR""","""27614-7002"""
"""533963""","""APPLE HOTEL LLC""","""11410 COMMON OAKS DR""","""27614-7002"""
"""539107""","""APPLE HOTEL, LLC""","""11410 COMMON OAKS DR""","""27614-7002"""
"""526390""","""CITYMARKET HOTEL DEVELOPMENT L…","""11410 COMMON OAKS DR""","""27614-7002"""
"""536039""","""CLA WEBSTER HOTEL OPERATORS LP""","""11410 COMMON OAKS DR""","""27614-7002"""
"""542016""","""CLA WEBSTER HOTEL OPERATORS, L…","""11410 COMMON OAKS DR""","""27614-7002"""
"""525826""","""CONCORD AZTEC BRICKELL LLC""","""11410 COMMON OAKS DR""","""27614-7002"""
"""532588""","""CONCORD AZTEC SPRINGFIELD LLC""","""11410 COMMON OAKS DRIVE""","""27614-7002"""


## Fragility review

Two views of the same idea: `single_edge_cluster` flags clusters (size >= 2) held together by exactly one qualifying edge. `edge_density` (`n_hash_ids / n_edges`) trends toward its spanning-tree ceiling (~1) the more fragile a cluster is, and drops well below 1 as redundant edges pile up — so among clusters big enough to matter, the highest `edge_density` values are the ones worth a second look.

In [14]:
print("single-edge clusters (size >= 2), largest first:")
matched.filter(pl.col("single_edge_cluster")).sort("n_primary_ids", descending=True).head(10)

single-edge clusters (size >= 2), largest first:


cluster_id,stable_cluster_id,n_primary_ids,n_hash_ids,n_edges,thin_edges,n_manual_edges,dedup_ratio,edge_density,single_edge_cluster,n_distinct_BorrowerAddress,n_distinct_BorrowerZip,n_distinct_BorrowerName
str,str,u32,u32,u32,u32,u32,f64,f64,bool,u32,u32,u32
"""21472""","""AUTO_1984266780526077003""",4,2,1,1,0,2.0,2.0,true,1,1,2
"""17534""","""AUTO_1590001380580071038""",4,2,1,1,0,2.0,2.0,true,1,1,2
"""74989""","""AUTO_9221673234174852983""",4,2,1,1,0,2.0,2.0,true,1,1,2
"""28773""","""AUTO_2733947412901352423""",4,2,1,1,0,2.0,2.0,true,1,1,2
"""61960""","""AUTO_6932406963793708063""",4,2,1,1,0,2.0,2.0,true,1,1,2
"""89377""","""AUTO_12814204426080811395""",4,2,1,1,0,2.0,2.0,true,1,1,2
"""67876""","""AUTO_7900406667628778937""",4,2,1,1,0,2.0,2.0,true,1,1,2
"""27424""","""AUTO_2585919310167107187""",4,2,1,1,0,2.0,2.0,true,1,1,2
"""53785""","""AUTO_5747571347374224573""",4,2,1,1,0,2.0,2.0,true,1,1,2


In [15]:
print("most tree-like (fragile) clusters among those with >= 5 rows:")
matched.filter(pl.col("n_primary_ids") >= 5).sort("edge_density", descending=True).head(10)

most tree-like (fragile) clusters among those with >= 5 rows:


cluster_id,stable_cluster_id,n_primary_ids,n_hash_ids,n_edges,thin_edges,n_manual_edges,dedup_ratio,edge_density,single_edge_cluster,n_distinct_BorrowerAddress,n_distinct_BorrowerZip,n_distinct_BorrowerName
str,str,u32,u32,u32,u32,u32,f64,f64,bool,u32,u32,u32
"""1574""","""AUTO_139818084006441859""",5,4,3,3,0,1.25,1.333333,false,2,1,3
"""6442""","""AUTO_559270743836675086""",5,4,3,3,0,1.25,1.333333,false,2,1,3
"""25774""","""AUTO_2413167760336862134""",5,4,3,3,0,1.25,1.333333,false,2,1,3
"""46110""","""AUTO_4745532397288988122""",5,4,3,3,0,1.25,1.333333,false,1,2,3
"""59""","""AUTO_5298395916481366""",5,4,3,3,0,1.25,1.333333,false,1,2,3
"""47306""","""AUTO_4895489255237488005""",5,4,3,3,0,1.25,1.333333,false,1,2,3
"""27562""","""AUTO_2602692218126689821""",5,4,3,3,0,1.25,1.333333,false,2,1,3
"""15990""","""AUTO_1441596459689610182""",5,4,3,3,0,1.25,1.333333,false,2,1,3
"""33571""","""AUTO_3269402443589915162""",5,4,3,3,0,1.25,1.333333,false,1,2,3


## `thin_edges`

Edges backed by exactly `k_required` signals (no corroboration beyond the minimum bar) vs edges confirmed by more. A cluster where every edge is thin is riskier than one with the same edge count but plenty of doubly-confirmed links.

In [16]:
matched.with_columns(
    thin_edge_share=pl.col("thin_edges") / pl.col("n_edges")
).filter(pl.col("n_primary_ids") >= 5).sort("thin_edge_share", descending=True).head(10)

cluster_id,stable_cluster_id,n_primary_ids,n_hash_ids,n_edges,thin_edges,n_manual_edges,dedup_ratio,edge_density,single_edge_cluster,n_distinct_BorrowerAddress,n_distinct_BorrowerZip,n_distinct_BorrowerName,thin_edge_share
str,str,u32,u32,u32,u32,u32,f64,f64,bool,u32,u32,u32,f64
"""2519""","""AUTO_221585595102524726""",56,48,910,910,0,1.166667,0.052747,false,3,1,45,1.0
"""1167""","""AUTO_104645805202085965""",50,49,997,997,0,1.020408,0.049147,false,3,1,45,1.0
"""1690""","""AUTO_149907957114869028""",49,49,602,602,0,1.0,0.081395,false,1,2,35,1.0
"""3464""","""AUTO_305120516945188409""",48,48,559,559,0,1.0,0.085868,false,1,2,41,1.0
"""4397""","""AUTO_388282515727979673""",45,43,903,903,0,1.046512,0.047619,false,1,1,43,1.0
"""6872""","""AUTO_597471602931765632""",42,41,820,820,0,1.02439,0.05,false,1,1,41,1.0
"""8972""","""AUTO_792011735008747999""",42,42,397,397,0,1.0,0.105793,false,3,2,27,1.0
"""16851""","""AUTO_1522889830373392479""",41,40,670,670,0,1.025,0.059701,false,2,1,39,1.0
"""867""","""AUTO_78811878241398370""",37,37,266,266,0,1.0,0.139098,false,5,1,23,1.0


In [18]:
records['cluster_id'].value_counts().sort('count', descending=True).head(10)

cluster_id,count
str,u32
"""2519""",56
"""1167""",50
"""1690""",49
"""3464""",48
"""4397""",45
"""6872""",42
"""8972""",42
"""16851""",41
"""867""",37


In [19]:
records.filter(pl.col("BorrowerAddress").is_in(["11410 COMMON OAKS DR", "7 PEARL CT"])).select("record_id", "BorrowerName", "BorrowerAddress", "BorrowerZip", "hash_id", "cluster_id", "stable_cluster_id").sort("BorrowerAddress")

record_id,BorrowerName,BorrowerAddress,BorrowerZip,hash_id,cluster_id,stable_cluster_id
str,str,str,str,u64,str,str
"""524418""","""CONCORD DAYTON HOTEL II LLC""","""11410 COMMON OAKS DR""","""27614-7002""",9182206913886793362,"""2519""","""AUTO_221585595102524726"""
"""525064""","""CONCORD SIERRA NLI WPB HOTEL L…","""11410 COMMON OAKS DR""","""27614-7002""",16809756557198628460,"""2519""","""AUTO_221585595102524726"""
"""525138""","""CONCORD DAYTON HOTEL II, LLC""","""11410 COMMON OAKS DR""","""27614-7002""",17061730511859445723,"""2519""","""AUTO_221585595102524726"""
"""525826""","""CONCORD AZTEC BRICKELL LLC""","""11410 COMMON OAKS DR""","""27614-7002""",9667878362800026858,"""2519""","""AUTO_221585595102524726"""
"""526332""","""CS HOTEL 30W46TH LLC""","""11410 COMMON OAKS DR""","""27614-7002""",15473266472985335431,"""2519""","""AUTO_221585595102524726"""
"""526390""","""CITYMARKET HOTEL DEVELOPMENT L…","""11410 COMMON OAKS DR""","""27614-7002""",12536980896534942630,"""2519""","""AUTO_221585595102524726"""
"""527579""","""OAKLAND FIFTH AVENUE HOTEL ASS…","""11410 COMMON OAKS DR""","""27614-7002""",12150294121372361915,"""2519""","""AUTO_221585595102524726"""
"""527660""","""CS30W46TH LLC""","""11410 COMMON OAKS DR""","""27614-7002""",7445973640698186408,"""2519""","""AUTO_221585595102524726"""
"""527714""","""ROSLYN O-S HOTEL PARTNERS LLC""","""11410 COMMON OAKS DR""","""27614-7002""",10969801539171558831,"""2519""","""AUTO_221585595102524726"""


In [20]:
address_out['records'].filter(pl.col("BorrowerAddress").is_in(["11410 COMMON OAKS DR", "7 PEARL CT"])).select("record_id", "BorrowerName", "BorrowerAddress", "BorrowerZip", "hash_id", "cluster_id", "stable_cluster_id").sort("BorrowerAddress")

record_id,BorrowerName,BorrowerAddress,BorrowerZip,hash_id,cluster_id,stable_cluster_id
str,str,str,str,u64,str,str
"""524418""","""CONCORD DAYTON HOTEL II LLC""","""11410 COMMON OAKS DR""","""27614-7002""",9182206913886793362,"""2519""","""ADDR_ALIAS_001"""
"""525064""","""CONCORD SIERRA NLI WPB HOTEL L…","""11410 COMMON OAKS DR""","""27614-7002""",16809756557198628460,"""2519""","""ADDR_ALIAS_001"""
"""525138""","""CONCORD DAYTON HOTEL II, LLC""","""11410 COMMON OAKS DR""","""27614-7002""",17061730511859445723,"""2519""","""ADDR_ALIAS_001"""
"""525826""","""CONCORD AZTEC BRICKELL LLC""","""11410 COMMON OAKS DR""","""27614-7002""",9667878362800026858,"""2519""","""ADDR_ALIAS_001"""
"""526332""","""CS HOTEL 30W46TH LLC""","""11410 COMMON OAKS DR""","""27614-7002""",15473266472985335431,"""2519""","""ADDR_ALIAS_001"""
"""526390""","""CITYMARKET HOTEL DEVELOPMENT L…","""11410 COMMON OAKS DR""","""27614-7002""",12536980896534942630,"""2519""","""ADDR_ALIAS_001"""
"""527579""","""OAKLAND FIFTH AVENUE HOTEL ASS…","""11410 COMMON OAKS DR""","""27614-7002""",12150294121372361915,"""2519""","""ADDR_ALIAS_001"""
"""527660""","""CS30W46TH LLC""","""11410 COMMON OAKS DR""","""27614-7002""",7445973640698186408,"""2519""","""ADDR_ALIAS_001"""
"""527714""","""ROSLYN O-S HOTEL PARTNERS LLC""","""11410 COMMON OAKS DR""","""27614-7002""",10969801539171558831,"""2519""","""ADDR_ALIAS_001"""


In [ ]:
records.filter(pl.col("cluster_id") == "2622").select("record_id", "BorrowerName", "BorrowerAddress", "BorrowerZip", "hash_id").unique(subset=["BorrowerName", "BorrowerAddress", "BorrowerZip"]).sort("BorrowerName")

record_id,BorrowerName,BorrowerAddress,BorrowerZip,hash_id
str,str,str,str,u64
"""646876""","""FRIEDWALD CENTER FOR DIALYSIS,…","""475 NEW HEMPSTEAD ROAD""","""10956""",6187453111763336785
"""607174""","""FRIEDWALD CENTER FOR REHABILIT…","""475 NEW HEMPSTEAD ROAD""","""10956""",18086546270986880475
"""660108""","""SUMMIT AT FRIEDWALD CARE CENTE…","""475 NEW HEMPSTEAD ROAD""","""10956""",10183094102775570983


In [ ]:
records.filter(pl.col("hash_id") == 12160495072962979024)

record_id,LoanNumber,DateApproved,SBAOfficeCode,ProcessingMethod,BorrowerName,BorrowerAddress,BorrowerCity,BorrowerState,BorrowerZip,LoanStatusDate,LoanStatus,Term,SBAGuarantyPercentage,InitialApprovalAmount,CurrentApprovalAmount,UndisbursedAmount,FranchiseName,ServicingLenderLocationID,ServicingLenderName,ServicingLenderAddress,ServicingLenderCity,ServicingLenderState,ServicingLenderZip,RuralUrbanIndicator,HubzoneIndicator,LMIIndicator,BusinessAgeDescription,ProjectCity,ProjectCountyName,ProjectState,ProjectZip,CD,JobsReported,NAICSCode,Race,Ethnicity,UTILITIES_PROCEED,PAYROLL_PROCEED,MORTGAGE_INTEREST_PROCEED,RENT_PROCEED,REFINANCE_EIDL_PROCEED,HEALTH_CARE_PROCEED,DEBT_INTEREST_PROCEED,BusinessType,OriginatingLenderLocationID,OriginatingLender,OriginatingLenderCity,OriginatingLenderState,Gender,Veteran,NonProfit,ForgivenessAmount,ForgivenessDate,hash_id,cluster_id
str,i64,str,i64,str,str,str,str,str,str,str,str,i64,i64,f64,f64,f64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,str,i64,str,str,str,str,str,str,f64,str,u64,str
"""529485""",2909558504,"""02/22/2021""",460,"""PPS""","""CONCORD AZTEC SPRINGFIELD LLC""","""11410 COMMON OAKS DR""","""Raleigh""","""NC""","""27614-7002""","""05/18/2022""","""Paid in Full""",60,100,472019.8,472019.8,0.0,"""Homewood Suites by Hilton""",19248,"""Synovus Bank""","""1148 Broadway""","""COLUMBUS""","""GA""","""31901-2429""","""U""","""N""","""N""","""Existing or more than 2 years …","""Raleigh""","""WAKE""","""NC""","""27614-7002""","""NC-02""",37.0,721110.0,"""Unanswered""","""Unknown/NotStated""",1.0,472016.8,null,null,null,null,null,"""Limited Liability Company(LLC…",19248,"""Synovus Bank""","""COLUMBUS""","""GA""","""Male Owned""","""Non-Veteran""",null,477451.26,"""04/21/2022""",12160495072962979024,"""4750"""
